In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from model import NSConfig, NsModel, CUR_DIR

In [2]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(f"{CUR_DIR}/model/tokenizer_fixed.json")

In [3]:
cfg = NSConfig()

device = "cuda:1"

model = NsModel(cfg)
model.to(device)

model.load_state_dict(
    torch.load(f"{CUR_DIR}/model/ns_model_v2_epoch_1_step_10w.bin", map_location=device, weights_only=True)
)
model.to(device)
model.eval()


NsModel(
  (emb): Embedding(32000, 768)
  (layers): ModuleList(
    (0-11): 12 x TransformerBlock(
      (rms_norm1): RMSNorm()
      (attn): GQAAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=False)
        (w_k): Linear(in_features=768, out_features=128, bias=False)
        (w_v): Linear(in_features=768, out_features=128, bias=False)
        (w_o): Linear(in_features=768, out_features=768, bias=False)
      )
      (rms_norm2): RMSNorm()
      (ffn): FeedForward(
        (fc1): Linear(in_features=768, out_features=2048, bias=False)
        (fc2): Linear(in_features=768, out_features=2048, bias=False)
        (fc3): Linear(in_features=2048, out_features=768, bias=False)
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_out): Linear(in_features=768, out_features=32000, bias=False)
)

In [7]:
prompt = "The capital of France is"

input_ids = tokenizer.encode(prompt).ids

eos_token_id = tokenizer.token_to_id("<eos>")
input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

In [8]:
# no kv-cache

max_output = 200

with torch.no_grad():

    
    for _ in range(max_output):

        logits = model.forward(input_ids)

        logits = logits[:, -1, :]

        # 贪心
        # next_token = torch.argmax(logits, dim=-1, keepdim=True)

        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        if next_token.item() == eos_token_id:
            print("-------------end-----------------------")
            break
        
        input_ids = torch.cat([input_ids, next_token], dim=1)




output = tokenizer.decode(input_ids.squeeze(0).cpu().numpy())
print(output)


 The capital of France is seperate from France and they have an outskier simulcast where they show up at least seven times or so. They make it to the Scottish border just as if nestling nowhere in the Summer or sit around the desert wonderland or never there has been finished living there.
Friedrich November 11, 1857, a listen in his salute to exciting achievements and achievements of The United States and eight other countries. However, his essay was scarcely nominated for the Peace Prize in France and the actual Academy Award in Grove, Switzerland. He was granted the worldwide Prix Europa Est Auteur’s Prix promoter, the part by which the only German regional composer of that year’s Festival, Geoffrey George, was inducted into the literary competition of the Golden Age's festival. He died in Letcherien, La Gaudour, no doubt twelve years later of his fame for his poetry, being one of the most eminent Gianni Dioras created through


In [6]:
# enable kvcache

max_output = 1500

with torch.no_grad():


    # 1. prefill
    logits, past_key_values = model.forward(input_ids, use_cache=True)

    generated = input_ids


    for _ in range(max_output):

        logits = logits[:, -1, :]


        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        generated = torch.cat([generated, next_token], dim=1)

        if next_token.item() == eos_token_id:
            print("-------------end-----------------------")
            break

        logits, past_key_values = model(next_token, past_key_values=past_key_values, use_cache=True)

    
output = tokenizer.decode(generated.squeeze(0).cpu().numpy())
print(output)


 the cat jump over the dog field.
Biopa wondered how she was remaining over the rock she started out fighting every week. I tried to contain my anger, I participated in as slams as I had this deluded me into using proxies if I needed to. Sometimes I have a tendency to run my adkins to me, I confess that when it comes to disciplinary calls in Australian dialect, both shy and incomplete. So tired of the oh so far more I started with remarks which I knw ing in between. Maybe when we took the boat, we stopped the singing.
That delighted the ears.
Instead I let them join us.
So decided to reminisce that the sea had a rather militant spirits.
Thanks to you for clearing any platitudes that a certain crook reacted by
as mere guides and words, perhaps I should bookmarkLast Blooming Agony and reply to T.R.Enflavin's story highlighting the amazing Kahalleys!
________________Bent on speaking in the newest Bubby sequel video I wrote at length with Waddy Eisenhower.
Since then, it's not always easy 

In [7]:
# inference with attention mask

prompts = [
    "the cat jump over the dog",
    "my name is sam, and i watch the movie on tv"
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)


In [32]:
input_tensor 

tensor([[  266,  3200,  3978,   629,   266,  3050,     1,     1,     1,     1,
             1,     1],
        [  518,  1562,   325,  6143,    15,   291,  1172,  2176,   266,  2966,
           324, 20199]], device='cuda:1')

In [33]:
attn_mask_tensor

tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:1')

In [ ]:
# no kv-cache + atten_mask

max_output = 200

with torch.no_grad():

    
    for _ in range(max_output):

        logits = model.forward(input_tensor, attention_mask=attn_mask_tensor)
        # logits = model.forward(input_tensor, attention_mask=None)

        logits = logits[:, -1, :]

        # 概率采样
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )
        
        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 the cat jump over the dog He has helped the organization raise all of its money. Vacation messenger services and Animators Services held their April 23 and May 4 shows dinner and Host cocktail at the Dragon Gardens from 6-9pm with 'S and Trio Sandwich' hosts Jennie Romy MacDonald, Mayor Clady and social mascot Jake Magua. DJ Brantley, managing Editor @kexaminer.com Audio How a Niconymous MCiner Runs Back to Bye Bird Smoke Out tour from thelinbrounds. And Babe Muyo Kingsley, How They Came Out 10 goals, mention seeing them as a quality 'nacho's last'….7-5, 512.
- 2008 W M Rochester SuperGMC 1R Short Shell BBQ 437 Bay Point SF
- 2011 W M Old Junction 4M Tower 2 Vessel ROD BRAFT Recordivan @ the booth Lancers on the East Bay Loop in W M Rochester
- 2006 Hight
----------------------------
 my name is sam, and i watch the movie on tv because he's in some danger i'll include him as your voice and i get it from the barber barber barber barbertype booth at the ladies hair party nearby. I still

In [41]:

prompts = [
    "the cat jump over the dog",
    "my name is sam, and i watch the movie on tv"
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)

In [43]:
input_tensor

tensor([[  266,  3200,  3978,   629,   266,  3050,     1,     1,     1,     1,
             1,     1,   354],
        [  518,  1562,   325,  6143,    15,   291,  1172,  2176,   266,  2966,
           324, 20199,   291]], device='cuda:1')

In [ ]:
# kv-cache + atten_mask

max_output = 200


with torch.no_grad():


    # 1. prefill
    logits, past_key_values = model.forward(input_tensor, use_cache=True, attention_mask=attn_mask_tensor)

    generated = input_ids


    for _ in range(max_output):

        logits = logits[:, -1, :]


        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)
        # NOTE: 使用kvcache时，只传入new token
        logits, past_key_values = model(next_token, past_key_values=past_key_values, use_cache=True, attention_mask=attn_mask_tensor)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 the cat jump over the dog Paws!! Yup it was the pup at the front of the dog and Bob at the back.
I think while the name is an old Norman Quinn (not being so new at that you complain!) (sort of in recent years but given the fact that they are still near serving their elderly sisters and adorable champs), I happened to notice an insert version of this little clip along with reference quizeroset horn from this version. The hair is a clear callation of Cats And Be Especially Happy in their Development??? I remember reading about the Mommy Sheet that Pamela had on hand. They gave Adolescent Mothers believe they and Amy couldn't get married, so at that point we get 100 comments about that. It's from last season, that better revision of Casper's MA, Joe in WA (GSA), who was one of the best Angel pics in the category (and LA), because we had some clips that wasn't even featured until Season 4. I read
----------------------------
 my name is sam, and i watch the movie on tv and i'm confused...